# 17 · ¿Conocer la quebrada agrega algo a conocer el día?

> **No es una alerta ni una predicción para decidir evacuaciones.** Mide cuánto del catálogo se explica con el historial de cada quebrada además del tiempo del día. Ante una emergencia llama al 115 (INDECI).

**Pregunta.** Con el catálogo mejorado (150 fechas, 24 quebradas con historial ubicado), ¿predecir *qué quebrada se activa qué día* mejora si se multiplica la probabilidad del día por la participación histórica de cada quebrada?

**Pares.** (quebrada, día de temporada de lluvias); positivo si la quebrada aparece en el catálogo ese día. La participación `π_i` de cada quebrada se aprende en las otras temporadas.

**Teorías (escritas antes de mirar las tablas de este cuaderno; los capítulos 17 y 19 daban resultados parciales con menos quebradas):**
- **Q1** `producto` (P(día) × π_i) tiene más precisión media que `solo_dia`, con lluvia observada y con intervalo que excluye 0.
- **Q2** lo mismo vale con el pronóstico GEFS a 24, 48 y 72 h.
- **Q3** usar la participación del régimen (Niño o no) mejora al `producto`.
- **Q4** las 5 quebradas de mayor participación cubren más activaciones que el azar (5/24 ≈ 21 %), también sin 2017.
- **Q5** nada de esto depende de 2017.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from illapa_research import quebradas_dia as QD

RAIZ = Path("..") if Path("../catalogo").exists() else Path(".")
if os.environ.get("ILLAPA_RECALCULAR"):
    R = QD.correr(RAIZ)  # unos minutos: reentrena el modelo diario dejando una temporada fuera
else:
    R = {k: pd.read_csv(RAIZ / "catalogo" / f"quebradas_dia_{k}.csv") for k in ("metricas", "diferencias", "cobertura")}
pd.set_option("display.width", 200)

## 1. Precisión media por modelo (Q1, Q2, Q3, Q5)

`ap_base` es la fracción de pares positivos; `solo_quebrada` ignora el tiempo del día; `producto_regimen` usa la participación de Niño o no Niño según la temporada.

In [ ]:
m = R["metricas"]
m[m["subconjunto"].isin(["todo", "sin 2017"])].round(4)[["entrada", "subconjunto", "modelo", "pares", "positivos", "ap", "ap_base", "auc", "logloss"]]

## 2. Diferencias con intervalo (remuestreo de temporadas)

`dif_ap` = AP(a) − AP(b); `excluye_cero` dice si el intervalo del 90 % deja fuera el 0.

In [ ]:
R["diferencias"].round(4)[["entrada", "a", "b", "dif_ap", "p5", "p95", "excluye_cero"]]

## 3. ¿Cubren las 5 quebradas con más historial las activaciones? (Q4)

Para cada temporada el top 5 usa la participación aprendida en las otras temporadas.

In [ ]:
R["cobertura"].round(3)

## Lectura

Ver `docs/21-quebradas-y-dia.md` para la interpretación y los límites. En una línea: con la lluvia observada, saber la quebrada mejora algo la predicción de pares y las 5 con más historial cubren más del doble de lo esperado; con pronósticos esa ventaja no se distingue de cero.